The [Vault CLI](building-and-testing-a-vault-cli.qmd) stores local secret values
in the system keyring. Its next small improvement is a history of supported
command use: which operation ran, which secret names it targeted, and whether
it succeeded.

That history is useful when a person or an agent invokes the CLI. It helps
answer questions such as whether a missing credential was recently removed or
which development session retrieved a particular name. The history needs to
exclude the credential values themselves.

This walkthrough follows the implementation in `src/atlas/vault_audit.py` and
its integration with `src/atlas/vault.py`. It assumes macOS and explains file
locking from the beginning. It is the second post in
[Building Atlas](../collections/building-atlas.qmd).

**1. Define what an event records.** The command supplies an operation, a list of
targeted names, an outcome, and an exit code. The writer adds a UTC timestamp,
process ID, and optional session label. A fictional record looks like this,
expanded onto several lines for readability:

```json
{
  "timestamp": "2026-10-28T06:00:00+00:00",
  "operation": "get",
  "secret_names": ["B2_DEV_KEY"],
  "outcome": "success",
  "exit_code": 0,
  "pid": 12345,
  "session_id": "experiment-1"
}
```

The real file, `.atlas/logs/vault.jsonl`, has one JSON object per line. It is easy
to inspect with ordinary file tools and can be read one record at a time.

The event object deliberately has no field for a value, stdin, a captured command
output, or an exception message:

```python
from dataclasses import dataclass, field


@dataclass
class AuditEvent:
    operation: str
    secret_names: list[str] = field(default_factory=list)
    outcome: str = "success"
    exit_code: int = 0
```

`default_factory=list` gives every event its own list. That prevents an event's
names from accidentally becoming shared mutable state.

`secret_names` describes the command's targets. For `check`, it contains all the
valid manifest names even if an early backend error prevents every lookup from
completing. It is not a sequence of individual backend-access receipts.

**2. Wrap commands in a context manager.** The command integration is small:

```python
with audit("get", name):
    value = _call_keyring(_get_backend().get_password, name)
    # Existing missing-secret and output handling follows here.
```

The context first prunes old records, creates an event, and yields control to the
command. When the command finishes, it classifies the result and saves the event.
The complete outcome-handling portion is:

```python
@contextmanager
def audit(operation: str, name: str | None = None) -> Generator[AuditEvent, None, None]:
    _update_log()
    event = AuditEvent(operation, [name] if name is not None else [])
    try:
        yield event
    except typer.Exit as error:
        event.exit_code = error.exit_code
        if error.exit_code:
            event.outcome = "failure"
        raise
    except (typer.Abort, KeyboardInterrupt):
        event.outcome = "cancelled"
        event.exit_code = 1
        raise
    except Exception:
        event.outcome = "failure"
        event.exit_code = 1
        raise
    finally:
        _update_log(event)
```

`@contextmanager` turns this generator into something that works with `with`.
Execution pauses at `yield event` while the command body runs. If the body raises
an exception, Python brings it back into the generator at that handoff point.
The handlers update the event and re-raise the exception, preserving the CLI's
normal error behavior. The `finally` block saves the outcome on ordinary exits
and handled exception paths.
[Python's context manager documentation](https://docs.python.org/3.13/library/contextlib.html#contextlib.contextmanager).

The `Generator` annotation describes the decorated function's generator: it
yields an `AuditEvent`, takes no meaningful sent-in value, and returns no final
value. The lock context used later has the same structure but yields `None`.

Typer can use `Exit(0)` for a successful command. The handler therefore checks the
exit code instead of treating every `Exit` as a failure. Declining deletion is
another special case: `rm` explicitly changes the event to `cancelled` and
returns normally, leaving exit code `0`.

The whole command path is:

```{mermaid}
%%| label: fig-vault-audit-lifecycle
%%| fig-cap: "The command runs between two short log updates. The file lock is released before the command body begins."
flowchart TD
    Start["Enter audit context"] --> Prune["Lock, prune old records, publish log, unlock"]
    Prune --> Event["Create AuditEvent"]
    Event --> Run["Run Vault command"]
    Run --> Outcome["Keep success or record failure / cancellation"]
    Outcome --> Save["Lock, prune, add outcome record, publish log, unlock"]
    Save --> Return["Return or propagate the original exception"]
```

There is no start record written before the command. A process terminated before
its finalizer runs can leave no outcome record. This is a local command history
with known coverage, rather than a complete record of all credential access.

**3. See why two writers need coordination.** A process is a running program.
Two terminals or two agent sessions can start separate Vault processes, each
with its own memory and its own copy of the log data it has read.

Suppose both processes read the same existing history. Process A adds event A
and saves it. Process B still has its older copy, so it adds event B and saves
that version. Event A disappears from the final file.

This is a **lost update**. The problem spans the whole read/modify/write sequence.
Protecting only the final write would still allow both writers to prepare
different replacements from the same old history.

The implementation makes that whole sequence exclusive. Each writer must
acquire the same OS lock before reading the log, and release it after publishing
the replacement.

```{mermaid}
%%| label: fig-vault-lock-sequence
%%| fig-cap: "An exclusive lock makes the second Vault process wait until the first has published its update."
sequenceDiagram
    participant A as Vault process A
    participant OS as macOS
    participant B as Vault process B
    A->>OS: Acquire exclusive lock
    OS-->>A: Granted
    B->>OS: Acquire the same lock
    Note over B: Waits inside flock
    Note over A: Read, prune, add event, replace log
    A->>OS: Release lock
    OS-->>B: Granted
    Note over B: Read the latest log and update it
    B->>OS: Release lock
```

The second process reads after it has acquired the lock, so its starting history
includes the first process's completed update.

**4. Acquire and release the macOS file lock.** The helper is small enough to
read in full:

```python
@contextmanager
def _lock(path: Path) -> Generator[None, None, None]:
    descriptor = os.open(path, os.O_CREAT | os.O_RDWR, 0o600)
    with os.fdopen(descriptor, "a+b") as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
        try:
            yield
        finally:
            fcntl.flock(lock.fileno(), fcntl.LOCK_UN)
```

`os.open()` creates the lock file if necessary and returns a **file descriptor**,
a small integer identifying an open file to the OS. `os.fdopen()` wraps that
descriptor in a Python file object. The `0o600` mode requests read/write access
for the owner and no permissions for the group or other ordinary users when the
file is created.

`fcntl.flock(..., LOCK_EX)` asks macOS for an exclusive lock. A competing request
for the same locked file normally waits until it can proceed. `LOCK_UN` releases
the lock. These calls coordinate cooperating processes; they do not prevent an
uncooperative program from modifying the file without requesting the lock.
[Apple's file-locking documentation](https://developer.apple.com/library/archive/documentation/System/Conceptual/ManPages_iPhoneOS/man2/flock.2.html).

The helper uses `finally` so an ordinary error inside the `with` block still
releases the lock. The outer `with` closes the Python file object too.

The lock file is `.atlas/logs/.vault.lock`. Its existence does not mean a process
currently holds the lock. macOS tracks the live lock state; the file stays in
place between commands.

Only log updates acquire this lock. The process releases it before reading a
credential, waiting for hidden input, or asking for deletion confirmation.
A person pausing at a prompt therefore does not hold up other log updates.

**5. Apply a rolling 30-day retention window.** Once the writer holds the lock,
it reads the current time in UTC and calculates the oldest timestamp it will
keep:

```python
now = datetime.now(UTC)
cutoff = now - RETENTION  # RETENTION = timedelta(days=30)
```

For each JSON line, it parses the timestamp and retains the record only when
this condition holds:

```python
timestamp.tzinfo is not None and cutoff <= timestamp <= now
```

The cutoff is inclusive. A record exactly 30 days old is retained at that
instant; an older record is removed. A timezone is required so the comparison
has a defined meaning. Future records are discarded too.

Malformed JSON, missing timestamps, and invalid timestamp values cannot satisfy
the retention rule, so cleanup drops those records. The log is rewritten with
only the retained history and, on the completion path, the new event.

This is cleanup on use. Every command prunes at entry and completion, but no
background service runs while the Vault is idle. Filesystem failures can also
prevent cleanup and generate a warning. The implementation does not promise
deletion at the exact expiration instant while no command is running.

**6. Publish a complete replacement file.** The writer does not truncate the
current log and then rebuild it in place. It writes the updated history into a
temporary file in the same directory:

```python
descriptor, temporary = tempfile.mkstemp(
    prefix=".vault-", suffix=".tmp", dir=directory
)
try:
    with os.fdopen(descriptor, "w", encoding="utf-8") as destination:
        for record in records:
            destination.write(json.dumps(record) + "\n")
    os.replace(temporary, path)
finally:
    Path(temporary).unlink(missing_ok=True)
```

`mkstemp()` creates a distinct temporary file with owner-only access on this
platform. Placing it beside the target keeps the replacement within the same
filesystem. The final `os.replace()` publishes the completed file atomically:
readers can open the old version or the completed replacement, rather than a
partially rewritten target.
[Temporary-file creation](https://docs.python.org/3.13/library/tempfile.html#tempfile.mkstemp),
[atomic replacement](https://docs.python.org/3.13/library/os.html#os.replace).

```{mermaid}
%%| label: fig-vault-atomic-replacement
%%| fig-cap: "The existing log remains available while the replacement is prepared. A separate lock file coordinates writers across replacements."
flowchart TD
    Lock["Hold .vault.lock"] --> Read["Read vault.jsonl"]
    Read --> Filter["Keep recent records and add the new event"]
    Filter --> Temp["Write a complete .vault-unique.tmp file"]
    Temp --> Replace["os.replace: publish as vault.jsonl"]
    Replace --> Release["Release .vault.lock"]
```

The exclusive lock prevents lost updates between writers. Atomic replacement
protects the visible file from a partial rewrite. These are separate properties.
Atomic replacement alone also does not promise that the newest event survives
an abrupt power loss; this implementation does not add a disk-sync protocol.

A separate lock file is important here. Locking the log itself and then
replacing it would leave the lock associated with the old file, while another
process could open the replacement. Keeping `.vault.lock` stable gives all
supported writers a common object to coordinate through. It should not be
deleted as routine cleanup.

**7. Keep runtime files local and failures visible.** All runtime files belong
under one directory:

```text
.atlas/logs/
  vault.jsonl
  .vault.lock
  .vault-*.tmp
```

The repository's `.gitignore` contains `/.atlas/logs/`, so it covers the log,
temporary files, and lock file together. The log directory is created with
mode `0o700`: its owner can list, modify, and traverse it. The new log files
have mode `0o600`. These permissions limit other ordinary OS users; an agent
running as the owner shares that user's access.

The log writer catches filesystem errors and emits a short warning to stderr:

```text
Warning: Could not update Vault audit log (PermissionError).
```

It includes the exception type and omits the raw exception message. A failure to
write this diagnostic history does not change the Vault command's result or
mix a warning into its stdout data.

A session label can be supplied explicitly:

```sh
ATLAS_VAULT_SESSION_ID=experiment-1 uv run at vault check
```

The writer stores at most 256 characters; an absent or empty label becomes
`null`. This is self-reported context. The process ID identifies the running
process, but neither field authenticates an agent or establishes human approval.
There are no signatures in this implementation.

**8. Test retention, privacy, and real competing processes.** The tests use
temporary directories, so they do not touch the developer's actual audit file.
The CLI fixture patches `atlas.vault_audit.ROOT_PATH` and clears the optional
session environment variable.

The retention test fixes the clock instead of waiting 30 days. Its seeded file
contains records just before the cutoff, exactly at the cutoff, at the current
time, in the future, and with invalid timestamps. It verifies cleanup inside
the audit block, proving that pruning happens before the command body. It then
advances the clock by 31 days and checks that the old history disappears.

Privacy tests exercise the actual CLI lifecycle and an exception whose message
contains a dummy credential. They verify that values do not enter the saved
records. Another test puts newlines and JSON-looking text in the name and session
label. JSON encoding keeps that event on one physical line rather than creating
fake entries.

The concurrency test launches four separate Python processes. Each creates
three events against the same temporary log directory. A thread pool starts
those subprocesses concurrently; the competing log writers are OS processes,
not merely tasks sharing one Python interpreter.

The assertions check every expected event, temporary-file cleanup, and the log's
permissions:

```python
records = [json.loads(line) for line in audit_log.read_text().splitlines()]
assert len(records) == 12
assert {record["secret_names"][0] for record in records} == {
    f"worker-{worker}-{index}"
    for worker in range(4)
    for index in range(3)
}
assert not list(audit_log.parent.glob("*.tmp"))
assert S_IMODE(audit_log.stat().st_mode) == 0o600
```

Finally, a CLI test places an ordinary file where the `.atlas` directory would
need to be. The resulting write failure must produce a warning on stderr while
`get` still returns the exact dummy credential on stdout. That checks the
chosen failure policy without relying on permission bits behaving identically
under every test runner's privileges.

Run the audit tests alone, or include them in the full suite:

```sh
uv run pytest tests/test_vault_audit.py -q
uv run pytest --cov=atlas --cov-branch --cov-report=term-missing
```

**9. Read the history within its coverage.** The recorded outcomes distinguish
normal completion, errors, and cancellation:

| Situation | Outcome | Exit code |
| --- | --- | --- |
| Normal command completion | `success` | `0` |
| Missing secret or missing entries in `check` | `failure` | `1` |
| Handled input or backend error | `failure` | `2` |
| User declines deletion | `cancelled` | `0` |
| Prompt abort or Ctrl+C | `cancelled` | `1` |

Only calls that enter the Vault wrappers receive these records. Calling the
keyring backend directly bypasses the log. A process running as the same OS user
can alter or delete the history, and the advisory lock does not stop it from
doing so. The log also records credential retrieval, not later requests made to
an API with that credential.

Provider permissions remain the control on those later actions. The audit log
adds a readable account of supported CLI use, with explicit retention and
concurrency behavior, while leaving credential authority with the keyring and
the external service.